# River forecasting with Optuna: RMSE baseline

This notebook is the single-output baseline. It uses a value head with differentiable RMSE loss and does not use spectrum loss, a spectrum head, or fractional loss.

## Load the implementation

The implementation is loaded from `forecast_river_data_optuna.py` so the notebook and script remain synchronized. This cell defines functions only; it does not run Optuna or neural-network training.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'data_lake' / 'config.json').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
PYTHON_DIR = PROJECT_ROOT / 'python'
if str(PYTHON_DIR) not in sys.path:
    sys.path.insert(0, str(PYTHON_DIR))

SCRIPT_PATH = PYTHON_DIR / 'forecast_river_data_optuna.py'
namespace = {'__name__': 'forecast_river_data_optuna_notebook', '__file__': str(SCRIPT_PATH)}
exec(compile(SCRIPT_PATH.read_text(encoding='utf-8'), str(SCRIPT_PATH), 'exec'), namespace)
globals().update({name: value for name, value in namespace.items() if not name.startswith('__')})
print('Loaded RMSE baseline:', SCRIPT_PATH)

/home/it_dash/Water-Analyzer-Ukr/.venv-linux/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
I0000 00:00:1791302516.223225   27278 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1791302516.270072   27278 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1791302517.842965   27278 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


Loaded RMSE baseline: /home/it_dash/Water-Analyzer-Ukr/python/forecast_river_data_optuna.py


## Load and validate data

MFDFA row 150 is assigned to target position 150. Spectrum labels are not required by this baseline.

In [2]:
ts_data, final_features_df, horizon = load_aligned_dataset()
print('Time series:', ts_data.shape)
print('Aligned model table:', final_features_df.shape)
print('First MFDFA index:', int(final_features_df['Index'].iloc[0]))
print('Primary loss: RMSE')

Time series: (410, 2)
Aligned model table: (260, 18)
First MFDFA index: 150
Primary loss: RMSE


## Fold preparation

Run this cell before tuning.

In [3]:
train_val_df = final_features_df.iloc[:-horizon].reset_index(drop=True)
test_df = final_features_df.iloc[-horizon:].reset_index(drop=True)
cv_folds = build_cv_folds(train_val_df)
print('Train/validation rows:', len(train_val_df))
print('Holdout rows:', len(test_df))
print('CV folds:', len(cv_folds))

Train/validation rows: 210
Holdout rows: 50
CV folds: 3


## Optuna tuning

Run this cell only when ready to train. Optuna minimizes inverse-scaled forecast RMSE.

In [4]:
study = optuna.create_study(
    direction='minimize',
    sampler=TPESampler(seed=SEED),
    pruner=HyperbandPruner(min_resource=10, max_resource=MAX_EPOCHS, reduction_factor=3),
    study_name='river_lstm_optuna',
)
study.optimize(lambda trial: objective(trial, cv_folds), n_trials=N_TRIALS)
best_params = study.best_trial.params
print('Best CV RMSE:', study.best_value)
print('Best parameters:', best_params)

[I 2026-10-06 12:02:29,215] A new study created in memory with name: river_lstm_optuna
E0000 00:00:1791302549.398689   27278 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
/home/it_dash/Water-Analyzer-Ukr/python/forecast_river_data_optuna.py:383: UserWarning: The reported value is ignored because this `step` 1 is already reported.
  trial.report(float(np.mean(fold_rmses)), step=fold_idx)
/home/it_dash/Water-Analyzer-Ukr/.venv-linux/lib/python3.12/site-packages/optuna_integration/tfkeras/tfkeras.py:57: UserWarning: The reported value is ignored because this `step` 0 is already reported.
  self._trial.report(float(current_score), step=epoch)
/home/it_dash/Water-Analyzer-Ukr/.venv-linux/lib/python3.12/site-packages/optuna_integration/tfkeras/tfkeras.py:57: UserWarning: The reported value is ignored because this `step` 1 is already reported.
  self._trial.report(float(current_score), step=epoch)
/home/it_dash/Water-Analyzer-Ukr/

/home/it_dash/Water-Analyzer-Ukr/python/forecast_river_data_optuna.py:383: UserWarning: The reported value is ignored because this `step` 3 is already reported.
  trial.report(float(np.mean(fold_rmses)), step=fold_idx)
[I 2026-10-06 12:02:39,790] Trial 0 finished with value: 0.9958915737540764 and parameters: {'lookback': 5, 'units': 16, 'dropout_rate': 0.02904180608409973, 'learning_rate': 0.005399484409787433, 'batch_size': 32, 'top_k': 4}. Best is trial 0 with value: 0.9958915737540764.
/home/it_dash/Water-Analyzer-Ukr/python/forecast_river_data_optuna.py:383: UserWarning: The reported value is ignored because this `step` 1 is already reported.
  trial.report(float(np.mean(fold_rmses)), step=fold_idx)
/home/it_dash/Water-Analyzer-Ukr/.venv-linux/lib/python3.12/site-packages/optuna_integration/tfkeras/tfkeras.py:57: UserWarning: The reported value is ignored because this `step` 0 is already reported.
  self._trial.report(float(current_score), step=epoch)
/home/it_dash/Water-Analyzer-

Best CV RMSE: 0.9213363361257407
Best parameters: {'lookback': 3, 'units': 48, 'dropout_rate': 0.26136641469099703, 'learning_rate': 0.0007162786999897341, 'batch_size': 32, 'top_k': 8}


## Final fit and artifact publication

The following cells fit the selected baseline and publish its CSV, metadata, plot, and lineage sidecars.

In [5]:
model, preds, actuals, selected_features = fit_final_model(
    train_val_df, test_df, best_params, cv_folds
)
stability = feature_stability(cv_folds, best_params['top_k'])
test_rmse = np.sqrt(mean_squared_error(actuals, preds))
print('Selected features:', selected_features)
print('Test RMSE:', test_rmse)
print(stability.to_string(index=False))

Selected features: ['Delta_H', 'H', 'Entropy', 'B', 'Kurtosis', 'Width', 'Skew', 'Delta_f_alpha']
Test RMSE: 0.8220735190447669
      Feature  Mean_Importance  Mean_Rank  Selection_Frequency
            H         0.003558   4.000000             1.000000
      Delta_H         0.002761   5.333333             1.000000
      Alpha_0         0.001729   6.000000             0.666667
            B         0.001675   8.000000             0.666667
         Mean        -0.001012   9.000000             0.666667
        Width        -0.002063  10.000000             0.666667
         beta         0.001817   6.666667             0.333333
         Skew         0.002264   8.000000             0.333333
     Kurtosis         0.001580   8.000000             0.333333
  Delta_alpha        -0.000173   8.333333             0.333333
        Slope         0.001756   9.333333             0.333333
   Volatility        -0.001874   9.666667             0.333333
        beta2        -0.001281  10.000000            

In [6]:
config_path = project_path('data_lake', 'config.json')
config_params = load_config(config_path)
indicator = config_params['INDICATOR']
river_name = config_params['RIVER_NAME_VALUE_ENG']
forecasting_folder = project_path(config_params['FORECASTING_FOLDER'])
curated_path = project_path(config_params['CURATED_FOLDER'], f'{river_name}_{indicator}_ts.csv')
features_path = project_path(config_params['FEATURES_FOLDER'], f"MFDFA_{river_name}_{indicator}_{config_params['MFDFA_WINDOW']}_window.csv")
artifacts = publish_forecast_artifacts(
    ts_data=ts_data, preds=preds, actuals=actuals, horizon=horizon,
    indicator=indicator, river_name=river_name, best_params=best_params,
    best_cv_rmse=study.best_value, test_rmse=test_rmse,
    selected_features=selected_features, stability=stability,
    forecasting_folder=forecasting_folder, config_path=config_path,
    input_files=[curated_path, features_path],
)
for artifact_type, artifact_path in artifacts.items():
    print(f'Published {artifact_type}: {artifact_path}')

Published csv: /home/it_dash/Water-Analyzer-Ukr/data_lake/forecasts/Forecasted_Dnipro_BSK5_10_days_LSTM.csv
Published metadata: /home/it_dash/Water-Analyzer-Ukr/data_lake/forecasts/Forecasted_Dnipro_BSK5_10_days_LSTM.json
Published metadata_lineage: /home/it_dash/Water-Analyzer-Ukr/data_lake/forecasts/Forecasted_Dnipro_BSK5_10_days_LSTM.json.lineage.json
Published plot: /home/it_dash/Water-Analyzer-Ukr/data_lake/forecasts/Forecasted_Dnipro_BSK5_10_days_LSTM.png
Published plot_lineage: /home/it_dash/Water-Analyzer-Ukr/data_lake/forecasts/Forecasted_Dnipro_BSK5_10_days_LSTM.png.lineage.json
Published csv_lineage: /home/it_dash/Water-Analyzer-Ukr/data_lake/forecasts/Forecasted_Dnipro_BSK5_10_days_LSTM.csv.lineage.json
